In [ ]:
from datetime import datetime
import numpy as np
import pandas as pd
import pycountry
import json
from IPython.display import display, Markdown
from matplotlib import pyplot as plt
from matplotlib.pyplot import cm

from emu_renewal.inputs import DATA_PATH, get_first_date_above_cov
from emu_renewal.inputs import get_worldbank_national_pop, get_country_mobility, get_standard_targets, get_row_proportions, get_all_var_data, get_country_var_data

In [ ]:
# Standard inputs
analysis_start = datetime(2020, 1, 1)
analysis_end = datetime(2022, 12, 31)
init_duration = 50
var_target_start_date = datetime(2021, 1, 1)
var_target_end_date = datetime(2024, 4, 1)
seed_duration = 10
date_format = "%d/%m/%Y"

In [ ]:
initial_countries = json.load(open(DATA_PATH / f"config/countries.json", "r"))
all_countries = initial_countries["admissions"] + initial_countries["occupancy"]

In [ ]:
country = "Chile"

In [ ]:
iso2 = pycountry.countries.lookup(country).alpha_2
iso3 = pycountry.countries.lookup(country).alpha_3
country_name = pycountry.countries.lookup(country).name
pop = get_worldbank_national_pop(iso3)
hosp_indicator, hosp_colour = ("Weekly new hospital admissions", "black") if iso2 in initial_countries["admissions"] else ("Daily hospital occupancy", "red")
cases_target, hosp_target, deaths_target, seroprev_target, init_data = get_standard_targets(
    iso2, analysis_start, analysis_end, init_duration, hosp_indicator,
)
display(Markdown(f"### {country_name}\n\nISO3: {iso3}\n"))
display(Markdown(f"population: {round(pop / 1e6, 3)} million\n"))
coverage_threshold = 0.25
if iso3 == "CHE":
    display(Markdown("Switzerland doesn't seem to have vaccination coverage data available in OWID"))
    coverage_threshold_time = datetime(2021, 3, 26)
else:
    coverage_threshold_time = get_first_date_above_cov(iso3, coverage_threshold)
display(Markdown(f"date that vaccination coverage passes {coverage_threshold * 100}%: {datetime.strftime(coverage_threshold_time, '%d/%m/%Y')}"))

In [ ]:
fig, axes = plt.subplots(6, 1, figsize=[8, 16], sharex=True)
fig.suptitle(country)
axes[0].plot(cases_target.index, cases_target, linewidth=0, marker=".")
axes[0].set_title("cases")
axes[1].plot(hosp_target.index, hosp_target, linewidth=0, marker=".", color=hosp_colour)
axes[1].set_title("hospitalisations")
plt.setp(axes[-1].xaxis.get_majorticklabels(), rotation=70)
axes[2].plot(deaths_target.index, deaths_target, linewidth=0, marker=".")
axes[2].set_title("deaths")
axes[3].plot(seroprev_target.index, seroprev_target, linewidth=0, marker=".")
axes[3].set_title("seroprevalence")
axes[3].set_ylim([0.0, 1.0])
mobility = get_country_mobility(iso3)
axes[4].plot(mobility.index, mobility)
all_var_data = get_all_var_data()
var_country_name = pycountry.countries.lookup(iso3).official_name if iso3 in ["CZE"] else pycountry.countries.lookup(iso3).name
var_data = get_country_var_data(all_var_data, var_country_name)
var_props = get_row_proportions(var_data)
plt.stackplot(var_props.index, var_props.T, colors=cm.turbo(np.linspace(0.0, 1.0, len(var_props.columns))))
fig.tight_layout()